#### Q. how langchain uses mcp servers?
- langchain uses `from langchain_mcp_adapters.client import MultiServerMCPClient` to instantiate mcp client
- then it get the tools for that mcp client
- bind the llm to those tools

In [1]:
import asyncio
from typing import Annotated, TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_mcp_adapters.client import MultiServerMCPClient

In [2]:
load_dotenv()

True

### using `playwright`

In [ ]:

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def route(s: State):
    return "tools" if getattr(s["messages"][-1], "tool_calls", None) else END

async def main():
    mcp = MultiServerMCPClient({
        "playwright": {"transport": "stdio", "command": "npx", "args": ["@playwright/mcp@latest"]}
    })
    tools = await mcp.get_tools()
    llm = ChatOpenAI(model="gpt-4.1-mini").bind_tools(tools)

    async def chat(s: State):
        return {"messages": [await llm.ainvoke(s["messages"])]}

    g = StateGraph(State)
    g.add_node("chat", chat)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "chat")
    g.add_conditional_edges("chat", route, {"tools": "tools", END: END})
    g.add_edge("tools", "chat")

    app = g.compile()
    out = await app.ainvoke({"messages": [HumanMessage(content="search google for current weather in NYC")]} )
    print(out["messages"][-1].content)

await main()


It looks like Google detected unusual traffic from my network and presented a CAPTCHA challenge that I cannot solve. Therefore, I'm unable to retrieve the current weather in NYC directly from Google at this moment.

Would you like me to try another weather information source or site?


### using `serpapi`

In [3]:
import os
from typing import Annotated, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_mcp_adapters.client import MultiServerMCPClient

In [5]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def route(s: State):
    return "tools" if getattr(s["messages"][-1], "tool_calls", None) else END

async def main():
    key = os.environ["SERPAPI_API_KEY"]
    mcp = MultiServerMCPClient({
        "serpapi": {"transport": "http", "url": f"https://mcp.serpapi.com/{key}/mcp"}
    })
    tools = await mcp.get_tools()
    llm = ChatOpenAI(model="gpt-4.1-mini").bind_tools(tools)

    async def chat(s: State): return {"messages": [await llm.ainvoke(s["messages"])]}

    g = StateGraph(State)
    g.add_node("chat", chat)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "chat")
    g.add_conditional_edges("chat", route, {"tools": "tools", END: END})
    g.add_edge("tools", "chat")

    out = await g.compile().ainvoke({
        "messages": [HumanMessage(content="search google for current weather in Calgary, Alberta")]
    })
    print(out["messages"][-1].content)


await main()

The current weather in Calgary, Alberta is -5°C with light snow showers. The humidity is 95%, precipitation chance is 21%, and the wind speed is 11 km/h. It is Monday at 10:00 p.m. in Calgary.

If you want details for the upcoming days:
- Monday: High 0°C, low -12°C, snow.
- Tuesday: Heavy snow storm, high -13°C, low -25°C.
- Wednesday: Partly sunny, high -21°C, low -27°C.
- Thursday: Partly sunny, high -18°C, low -27°C.
- Friday: Cloudy, high -9°C, low -21°C.

Let me know if you need more specific details or forecasts!
